# 🌍 Data Visualization Part 6: Interactive Dashboards & Geospatial Maps
**From Static Charts to Production Web Apps and Global Mapping**

Welcome to Part 6! You know how to build beautiful charts, but how do you share them with a non-technical stakeholder? You build a **Dashboard**. How do you visualize location-based data? You build a **Map**. 

In this final masterclass, we will learn:
1. **Plotly Dash:** Turning Python code into interactive web applications.
2. **Plotly Geospatial:** Mapping data globally using ScatterGeo and Choropleth.
3. **Folium:** Creating highly customizable, tile-based interactive maps.

Every concept follows our **Mastery Framework**:

`Theory → Mental Model → Diagram → Code → Mistakes → Interview → Practice (Levels 1-5)`


In [1]:
# ========================================== 
# 📦 DEPENDENCY INSTALLATION (Run this first in Colab)
# ========================================== 
!pip install -q folium dash jupyter-dash

# ==========================================
# 🛠️ SETUP & ENVIRONMENT
# ==========================================
import pandas as pd
import numpy as np
import plotly.express as px
import folium

# Dash imports (Modern syntax)
from dash import Dash, html, dcc, Input, Output, callback
# JupyterDash allows Dash apps to run directly inside Google Colab!
from jupyter_dash import JupyterDash 

print("✅ Dash, Plotly, and Folium loaded successfully!")

# 1. Gapminder Dataset (For Maps)
df_gapminder = px.data.gapminder()

# 2. Global Cities Dataset (For Mapping Coordinates)
np.random.seed(42)
cities_data = pd.DataFrame({
    'city': ['Tokyo', 'New York', 'London', 'Paris', 'Sydney', 'Mumbai', 'Rio', 'Cairo'],
    'lat': [35.68, 40.71, 51.50, 48.85, -33.86, 19.07, -22.90, 30.04],
    'lon': [139.69, -74.00, -0.12, 2.35, 151.20, 72.87, -43.17, 31.23],
    'population': [37, 8.4, 9, 2.1, 5.3, 20, 6.7, 9.5], # in millions
    'continent': ['Asia', 'N. America', 'Europe', 'Europe', 'Oceania', 'Asia', 'S. America', 'Africa']
})

print("✅ Datasets Ready!")


✅ Dash, Plotly, and Folium loaded successfully!
✅ Datasets Ready!


# 📘 Module 1: Introduction to Plotly Dash (Layouts)

## 🧠 1. Theory
**Plotly Dash** is a Python framework for building analytical web applications. It connects the power of Plotly charts with the interactivity of HTML/React, without you needing to know JavaScript.

## 💡 2. Mental Model
Think of Dash as **Lego Blocks**. 
- `html.*` components are the bricks (Divs, Headers, Tables).
- `dcc.*` components are the interactive pieces (Dropdowns, Sliders, Graphs).
- You snap them together to build a `layout`.

## 📊 3. Visual Diagram
```text
Dash App Structure
│
├── app.layout (The Blueprint)
│      ├── html.H1("My Dashboard")
│      ├── dcc.Dropdown(...)
│      └── dcc.Graph(...)
│
└── app.run() (The Engine that renders it)
```

## 💻 4. Code Example


In [2]:
# Initialize the app for Colab

app = Dash(__name__)

app.layout = html.Div([
    html.H1(
        "My First Dash App",
        style={
            'textAlign': 'center',
            'color': '#0099ff'
        }
    ),

    html.P("This is a static layout. No interactivity yet!"),

    dcc.Graph(
        figure=px.scatter(
            cities_data,
            x='population',
            y='lat',
            color='continent'
        )
    )
])

app.run(mode='inline', port=8050)

## ⚠️ 5. Common Mistakes
**The Blank Screen Trap:** Forgetting to call `app.run_server(mode='inline')` in Colab. The app is built in memory, but the server never starts to render it in the notebook output.

## 🎤 6. Interview Question
**Q:** What is the difference between `html.Div` and `dcc.Graph` in Dash?
**A:** `html.Div` is a standard HTML container used for structuring and styling the page layout (like a `<div>` in web development). `dcc.Graph` is a Dash Core Component specifically designed to render interactive Plotly figures.

## 🎯 7. Practice Tasks
*   **Level 1 (Easy):** Create a Dash app with an `H2` header that says "Sales Dashboard".
*   **Level 2 (Medium):** Add a `dcc.Markdown` component that renders a bolded bullet-point list.
*   **Level 3 (Hard):** Add two `dcc.Graph` components side-by-side using `html.Div(style={'display': 'flex'})`.
*   **Level 4 (Expert):** Style the background color of the main `html.Div` to light gray.
*   **Level 5 (Challenge):** Embed a Plotly Express Pie chart of the `tips` dataset into the layout.


In [3]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 1
# ==========================================
# Example Execution
# Create Dash App
app = JupyterDash(__name__)

#Layout
app.layout = html.Div([

    html.H1(
        "Global Cities Dashboard",
        style={
            "textAlign": "center",
            "fontFamily": "Arial"
        }
    ),

    html.Hr(),
    
    html.P(
        "Exploring population and coordinates.", 
        style={'textAlign': 'center'}
    ),
    
    dcc.Graph(
        figure=px.bar(
            cities_data, 
            x='city', 
            y='population', 
            color='continent', 
            title="Population by City"
        )
    )
])

app.run(mode='inline', port=8051)

# --- YOUR TURN ---
# Level 1-5: Build your own custom static layout!


d:\Projects\personalNotes\.venv\Lib\site-packages\jupyter_dash\jupyter_app.py:103: UserWarning: JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.
  super(JupyterDash, self).__init__(name=name, **kwargs)


# 📘 Module 2: Dash Callbacks (Interactivity)

## 🧠 1. Theory
A layout is just a static picture. **Callbacks** bring it to life. A callback is a Python function that automatically updates an `Output` component whenever an `Input` component changes.

## 💡 2. Mental Model
**The Waiter and the Kitchen.**
- **Input:** The customer (user) changes the Dropdown (orders a steak).
- **Callback:** The Waiter (Python function) takes the order to the kitchen.
- **Output:** The Kitchen (Plotly) cooks the steak and returns the updated Graph.

## 📊 3. Visual Diagram
```text
[ Dropdown (Input) ] 
       │ triggers
       ▼
[ Python Function ] ──(Filters Data)──> [ dcc.Graph (Output) ]
```

## 💻 4. Code Example


In [4]:
# Create Dash App
app = JupyterDash(__name__)

# Layout
app.layout = html.Div([

    html.H2("Filter by Continent"),
    
    # The Input: A dropdown menu
    dcc.Dropdown(
        id='continent-dropdown',
        options=[{'label': c, 'value': c} for c in cities_data['continent'].unique()],
        value='Asia' # Default value
    ),
    # The Output: The graph that will update
    dcc.Graph(id='population-graph')
])

# The Callback: Connects Input to Output
@callback(
    Output('population-graph', 'figure'),
    Input('continent-dropdown', 'value')
)
def update_graph(selected_continent):
    filtered_df = cities_data[cities_data['continent'] == selected_continent]
    fig = px.bar(filtered_df, x='city', y='population', title=f"Cities in {selected_continent}")
    return fig

app.run(mode='inline', port=8052)


d:\Projects\personalNotes\.venv\Lib\site-packages\jupyter_dash\jupyter_app.py:103: UserWarning: JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.
  super(JupyterDash, self).__init__(name=name, **kwargs)


## ⚠️ 5. Common Mistakes
**ID Mismatch:** The `id` in the `Output`/`Input` of the `@callback` decorator *must exactly match* the `id` in the layout components. If they don't match, the app will crash or do nothing.

## 🎤 6. Interview Question
**Q:** Can a single Dash Callback have multiple `Input` components?
**A:** Yes! The callback function will trigger whenever *any* of the listed `Input` components change. The function arguments must match the order of the `Input` definitions.

## 🎯 7. Practice Tasks
*   **Level 1 (Easy):** Add a `dcc.Slider` to filter the Gapminder dataset by `year` (1952 to 2007).
*   **Level 2 (Medium):** Add a `dcc.RadioItems` to toggle the chart type between 'Bar' and 'Scatter'.
*   **Level 3 (Hard):** Create a dashboard with *two* dropdowns (Continent and Metric) that filter the data simultaneously.
*   **Level 4 (Expert):** Use `State` to read the value of a text box only when a "Submit" button is clicked.
*   **Level 5 (Challenge):** Build a fully interactive Gapminder dashboard with a dropdown for X-axis, Y-axis, and Color.


In [5]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 2
# ==========================================
# Example Execution: Interactive Gapminder App
app = JupyterDash(__name__)

app.layout = html.Div([
    html.H1("Interactive Gapminder Explorer"),
    
    html.Div([
        html.Label("Select Year:"),
        dcc.Slider(id='year-slider', min=1952, max=2007, step=5, value=2007,
                   marks={str(year): str(year) for year in range(1952, 2008, 5)})
    ]),
    
    dcc.Graph(id='gapminder-scatter')
])

@callback(
    Output('gapminder-scatter', 'figure'),
    Input('year-slider', 'value')
)
def update_scatter(selected_year):
    df_year = df_gapminder[df_gapminder['year'] == selected_year]
    fig = px.scatter(df_year, x='gdpPercap', y='lifeExp', size='pop', color='continent',
                     hover_name='country', log_x=True, size_max=55,
                     title=f"Global Development in {selected_year}")
    return fig

app.run(mode='inline', port=8053)

# --- YOUR TURN ---
# Level 1-5: Add more interactivity to this app!


# 📘 Module 3: Geospatial Maps (Plotly Express)

## 🧠 1. Theory
Plotly Express has built-in functions to map data directly onto a 3D globe or a 2D flat map.
- **`scatter_geo`**: Plots points (cities, earthquakes) using Lat/Lon.
- **`choropleth`**: Colors entire regions (countries, states) based on a value.

## 💡 2. Mental Model
- **Scatter Geo:** Dropping pins on a digital globe.
- **Choropleth:** Painting countries on a map with a heat-based brush.

## 📊 3. Visual Diagram
```text
Scatter Geo:               Choropleth:
🌍 * (Tokyo)               🌍 [USA: Dark Blue]
   * (NY)                     [CAN: Light Blue]
   * (Lon)                    [MEX: Med Blue]
```

## 💻 4. Code Example


In [6]:
# 1. Scatter Geo (Plotting Cities)
fig_scatter = px.scatter_geo(cities_data, lat='lat', lon='lon', 
                             color='continent', size='population',
                             hover_name='city', projection="natural earth",
                             title="Major Global Cities")
fig_scatter.show()

# 2. Choropleth (Plotting Country Metrics)
fig_choro = px.choropleth(df_gapminder, locations="iso_alpha", 
                          color="lifeExp", hover_name="country", 
                          animation_frame="year", color_continuous_scale="Plasma",
                          range_color=(20, 80),
                          title="Global Life Expectancy Over Time")
fig_choro.show()


## ⚠️ 5. Common Mistakes
**Missing ISO Codes:** For `choropleth`, Plotly needs standard 3-letter country codes (ISO_ALPHA) to know which polygon to color. If your dataset only has "United States", you must map it to "USA" first, or Plotly will leave the map blank.

## 🎤 6. Interview Question
**Q:** What is the difference between `scatter_geo` and `choropleth`?
**A:** `scatter_geo` plots exact point coordinates (Lat/Lon) as markers. `choropleth` uses predefined geographic boundaries (like country borders) and fills the entire area with a color based on a data value.

## 🎯 7. Practice Tasks
*   **Level 1 (Easy):** Create a `scatter_geo` map of the `cities_data`, coloring by `population`.
*   **Level 2 (Medium):** Change the `projection` to `"orthographic"` (creates a 3D globe view).
*   **Level 3 (Hard):** Create a `choropleth` of Gapminder showing `gdpPercap` for the year 2007.
*   **Level 4 (Expert):** Add an `animation_frame="year"` to the Gapminder GDP choropleth to watch it evolve.
*   **Level 5 (Challenge):** Filter the Gapminder dataset to only show Asia, and plot a Scatter Geo of Asian countries sized by population.


In [7]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 3
# ==========================================
# Example Execution: Animated Choropleth
fig_choro = px.choropleth(df_gapminder, locations="iso_alpha", color="lifeExp", 
                          hover_name="country", animation_frame="year", 
                          color_continuous_scale="Viridis", projection="natural earth",
                          title="Evolution of Global Life Expectancy")
fig_choro.show()

# --- YOUR TURN ---
# Level 1-5: Master Plotly Geospatial mapping!


# 📘 Module 4: Advanced Maps with Folium

## 🧠 1. Theory
While Plotly is great for data-centric maps, **Folium** (built on Leaflet.js) is the industry standard for **tile-based maps** (like Google Maps). It allows you to zoom into street level, add custom markers, and draw shapes.

## 💡 2. Mental Model
Folium is like having an **empty digital map** in Python. You create the map, then you "drop pins" (Markers), "draw circles" (CircleMarkers), or "color neighborhoods" (Choropleth) onto it.

## 📊 3. Visual Diagram
```text
1. Create Map Object (The Canvas)
   ↓
2. Add Tile Layer (The Background Map)
   ↓
3. Add Markers/Popups (The Data Pins)
   ↓
4. Render Map
```

## 💻 4. Code Example


In [11]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 4
# ==========================================
# Example Execution: Interactive City Map
m = folium.Map(location=[20, 0], zoom_start=2, tiles="CartoDB positron")

for i, row in cities_data.iterrows():
    # Color coding based on population
    color = 'red' if row['population'] > 15 else 'blue'
    
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=row['population'] * 1.5, # Size based on population
        popup=f"{row['city']}: {row['population']} Million",
        color=color,
        fill=True,
        fill_color=color
    ).add_to(m)

folium.LayerControl().add_to(m)
display(m)

# --- YOUR TURN ---
# Level 1-5: Build your own custom Folium map!

# 🏢 Mini Project: The Global Real Estate Dashboard

## 🚀 Capstone Challenge: End-to-End Dashboard
Combine everything you've learned in Part 6 to build a mini web application.

**Requirements:**
1. **Layout:** Create a Dash app with a title, a dropdown to select a `continent`, and a graph area.
2. **Map Integration:** The graph area should display a `px.scatter_geo` map of the cities in the selected continent.
3. **Interactivity:** When the user changes the dropdown, the map should instantly re-render to show only the cities in that continent.
4. **Styling:** Center the title and add a nice background color to the dashboard.


In [14]:
# ==========================================
# 🏢 MINI PROJECT SOLUTION
# ==========================================
app = JupyterDash(__name__)

app.layout = html.Div(style={'backgroundColor': '#f4f6f9', 'padding': '20px'}, children=[
    html.H1("Global City Explorer Dashboard", style={'textAlign': 'center', 'color': '#2c3e50'}),
    
    html.Div([
        html.Label("Select Continent:", style={'fontWeight': 'bold', 'fontSize': '18px'}),
        dcc.Dropdown(
            id='map-continent-dropdown',
            options=[{'label': 'All', 'value': 'All'}] + 
                    [{'label': c, 'value': c} for c in cities_data['continent'].unique()],
            value='All',
            style={'width': '50%', 'margin': 'auto'}
        )
    ], style={'textAlign': 'center', 'marginBottom': '20px'}),
    
    dcc.Graph(id='geo-map-output', style={'backgroundColor': 'white', 'borderRadius': '10px'})
])

@callback(
    Output('geo-map-output', 'figure'),
    Input('map-continent-dropdown', 'value')
)
def update_map(selected_continent):
    if selected_continent == 'All':
        df_map = cities_data
    else:
        df_map = cities_data[cities_data['continent'] == selected_continent]
        
    fig = px.scatter_geo(df_map, lat='lat', lon='lon', color='continent',
                         size='population', hover_name='city', 
                         projection="natural earth",
                         title=f"Cities in {selected_continent}")
    fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    return fig

app.run(debug=True, mode='inline', port=8054)


d:\Projects\personalNotes\.venv\Lib\site-packages\jupyter_dash\jupyter_app.py:103: UserWarning: JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.
  super(JupyterDash, self).__init__(name=name, **kwargs)


KeyError: 'continent'


```

---

### 📝 [Markdown Cell]
```markdown
# 🎉 Congratulations! You are a Data Visualization Master!
You have completed the entire **Data Visualization Masterclass Series (Parts 1-6)**.

### 🏆 What You've Conquered:
✅ **Part 1:** Matplotlib Basics (Line, Scatter, Bar, Hist, Pie)
✅ **Part 2:** Advanced Matplotlib (Subplots, 3D, Annotations, Pandas Plot)
✅ **Part 3:** Plotly Express & Graph Objects (Interactive Web Charts)
✅ **Part 4:** Seaborn Basics (Relational, Distribution, Matrix)
✅ **Part 5:** Seaborn Advanced (Categorical, Regression, FacetGrids)
✅ **Part 6:** Dashboards & Maps (Plotly Dash, Geospatial, Folium)

### 🚀 Next Steps in Your Data Journey:
You now possess the visualization skills of a **Lead Data Scientist**. 
To complete your toolkit, move on to:
1. **Storytelling with Data:** Learning the psychology of how executives read charts.
2. **Machine Learning:** Using Scikit-Learn to predict the trends you just visualized.
3. **Cloud Deployment:** Hosting your Dash apps on Heroku, AWS, or Streamlit Cloud so the world can see them!

*Happy Visualizing! 📊✨*
```